# Chapter 2: The Simple Regression Model

Covers simple OLS estimation, fitted values and residuals, goodness of fit (R²), nonlinearities, and Monte Carlo verification of OLS properties.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import wooldridge as woo
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 2.1 Simple OLS Regression

In [ ]:
df = woo.dataWoo('wage1')

# OLS via statsmodels
model = smf.ols('wage ~ educ', data=df).fit()
print(model.summary())

## 2.2 Coefficients, Fitted Values, and Residuals

In [ ]:
print(f"Intercept (β₀): {model.params['Intercept']:.4f}")
print(f"Slope (β₁):     {model.params['educ']:.4f}")
print(f"\nFirst 5 fitted values:\n{model.fittedvalues.head()}")
print(f"\nFirst 5 residuals:\n{model.resid.head()}")
print(f"\nSum of residuals ≈ 0: {model.resid.sum():.6f}")

## 2.3 Goodness of Fit

In [ ]:
print(f"R-squared:  {model.rsquared:.4f}")
print(f"Adj. R²:    {model.rsquared_adj:.4f}")
print(f"SSR (resid): {model.ssr:.2f}")
print(f"SST (total): {(df['wage'] - df['wage'].mean()).pow(2).sum():.2f}")

## 2.4 Nonlinearities — Log Transformation

In [ ]:
df['log_wage'] = np.log(df['wage'])
model_log = smf.ols('log_wage ~ educ', data=df).fit()
print(model_log.summary())
print(f"\nInterpretation: 1 extra year of education → "
      f"{model_log.params['educ']*100:.2f}% higher wage")

## OLS Regression Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Level
axes[0].scatter(df['educ'], df['wage'], alpha=0.3, color='steelblue')
axes[0].plot(df['educ'], model.fittedvalues, 'r-', lw=2)
axes[0].set_xlabel('Education'); axes[0].set_ylabel('Wage')
axes[0].set_title('Wage ~ Education (Level)')

# Log
axes[1].scatter(df['educ'], df['log_wage'], alpha=0.3, color='darkorange')
axes[1].plot(df['educ'], model_log.fittedvalues, 'r-', lw=2)
axes[1].set_xlabel('Education'); axes[1].set_ylabel('Log(Wage)')
axes[1].set_title('Log(Wage) ~ Education')

plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_simple_ols.png", dpi=150)
plt.show()

## 2.7 Monte Carlo Simulation of OLS

In [ ]:
np.random.seed(123)
n, n_sims = 100, 2000
true_b0, true_b1, sigma = 1.0, 0.5, 2.0

betas = []
for _ in range(n_sims):
    x = np.random.uniform(0, 10, n)
    u = np.random.normal(0, sigma, n)
    y = true_b0 + true_b1 * x + u
    temp_df = pd.DataFrame({'y': y, 'x': x})
    m = smf.ols('y ~ x', data=temp_df).fit()
    betas.append(m.params['x'])

plt.figure(figsize=(8, 4))
plt.hist(betas, bins=50, edgecolor='black', color='steelblue', density=True)
plt.axvline(true_b1, color='red', lw=2, label=f'True β₁={true_b1}')
plt.axvline(np.mean(betas), color='green', lw=2, linestyle='--',
            label=f'Mean β̂₁={np.mean(betas):.3f}')
plt.legend(); plt.title('Monte Carlo: Sampling Distribution of β̂₁')
plt.savefig(f"{output_dir}/fig2_monte_carlo_ols.png", dpi=150)
plt.show()
print(f"Bias: {np.mean(betas) - true_b1:.4f}")
print(f"Std Dev of β̂₁: {np.std(betas):.4f}")